# Lab 14: Path-integral MCMC

We'll sample paths for the quantum harmonic oscillator, then use those paths to recover its position probability density.

The main comparison today: **sampling error versus time-step error**.

PHYS 142/242 · Friday, November 13, 2026

## Set up the notebook

Run the cells in order. Each simulation below starts a new chain from a stated seed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit

plt.rcParams.update({
    "font.size": 12, "figure.figsize": (6.2, 3.6),
    "figure.dpi": 110, "figure.constrained_layout.use": True,
})
M, OMEGA, HBAR = 1.0, 1.0, 1.0
TAU = 30.0

NumPy holds the paths; Matplotlib draws them. Numba compiles the repeated updates. We use units with $m=\omega=\hbar=1$.

## The result we're aiming for

For the ground state of the harmonic oscillator,

$$\rho_0(x)=|\psi_0(x)|^2=\sqrt{\frac{m\omega}{\pi\hbar}}\,e^{-m\omega x^2/\hbar},
\qquad \langle x^2\rangle_0=\frac{\hbar}{2m\omega}=\frac12.$$

We'll use this known answer to check the simulation.

In [ ]:
x_grid = np.linspace(-3, 3, 301)
rho_ground = np.sqrt(M * OMEGA / (np.pi * HBAR)) * np.exp(
    -M * OMEGA * x_grid**2 / HBAR)
fig, ax = plt.subplots()
ax.plot(x_grid, rho_ground, color="#007c91", lw=2)
ax.set(xlabel="Position x", ylabel="Probability density")
plt.show()

The histogram should approach this curve as the total imaginary time becomes long and the time step becomes small.

## What is one configuration?

In Lab 11, one configuration was a whole spin lattice. Here it is a whole path:

$$x=(x_0,x_1,\ldots,x_{N_\tau-1}),\qquad \tau_i=i\,\Delta\tau.$$

The $x_i$ are **continuous positions**. Only imaginary time is on a grid. One Monte Carlo update changes a position on this path; it does not advance physical time.

## Close the path

We impose $x_{N_\tau}=x_0$. The first and last stored points are neighbors.

In [ ]:
dt = 1.0
n_tau = int(round(TAU / dt))
tau_grid = np.arange(n_tau + 1) * dt
example_path = 0.8 * np.sin(2 * np.pi * np.arange(n_tau) / n_tau)
fig, ax = plt.subplots()
ax.plot(tau_grid, np.r_[example_path, example_path[0]], "o-", ms=3)
ax.set(xlabel=r"Imaginary time $\tau$", ylabel="Position x")
plt.show()

The final point repeats the first for drawing only. The array stores $N_\tau$ coordinates, with no duplicate endpoint.

## Why a long imaginary-time interval?

A periodic imaginary-time path samples the thermal trace. Excited states are suppressed by $e^{-\tau_{\rm total}(E_n-E_0)/\hbar}$.

For this oscillator, $\omega\tau_{\rm total}=30$, so the continuum thermal result is effectively the ground-state result.

That does **not** remove the error from a finite $\Delta\tau$. We'll test that separately.

## Assign a weight to each path

$$P(x)\propto e^{-S_E(x)/\hbar},$$

$$S_E(x)=\sum_{i=0}^{N_\tau-1}\left[
\frac{m(x_{i+1}-x_i)^2}{2\Delta\tau}
+\Delta\tau\,\frac{m\omega^2 x_i^2}{2}\right].$$

The first term penalizes jumps between neighboring times. The second penalizes large positions. We evaluate the potential at the **left endpoint**, as in Assignment 3 and Javier's corrected lab.

## One link, then the whole path

Keep the action in small functions so we can check each piece.

In [ ]:
@njit
def action(x_left, x_right, dt):
    kinetic = 0.5 * M * (x_right - x_left)**2 / dt
    potential = 0.5 * M * OMEGA**2 * x_left**2 * dt
    return kinetic + potential

@njit
def total_action(path, dt):
    result = 0.0
    for i in range(path.size):
        result += action(path[i], path[(i + 1) % path.size], dt)
    return result

The modulo operation closes the last link. Each link and each potential term are counted once.

## Change one point

Propose $x_i'=x_i+u$, with $u$ uniform on $[-h,h]$.

Only the two links touching $i$ change. We can compute $\Delta S_E$ without summing the entire path.

In [ ]:
i = n_tau // 2
proposal = example_path.copy()
proposal[i] += 0.65
fig, ax = plt.subplots()
ax.plot(tau_grid, np.r_[example_path, example_path[0]], "o-", label="Current")
ax.plot(tau_grid, np.r_[proposal, proposal[0]], "o--", label="Proposed")
ax.set(xlim=((i - 3) * dt, (i + 3) * dt),
       xlabel=r"Imaginary time $\tau$", ylabel="Position x")
ax.legend()
plt.show()

Everything outside those two links cancels when we subtract the old action from the proposed action.

## Compute the local action change

Replace the two old links by the two proposed links.

In [ ]:
@njit
def delta_action(path, x_prime, i, dt):
    left = path[(i - 1) % path.size]
    right = path[(i + 1) % path.size]
    new = action(left, x_prime, dt) + action(x_prime, right, dt)
    old = action(left, path[i], dt) + action(path[i], right, dt)
    return new - old

This works at $i=0$ and at the last stored point because both neighbor indices wrap around.

## Check the local calculation

Recompute the full action once and compare. This catches missing links and periodic-boundary mistakes.

In [ ]:
local_change = delta_action(example_path, proposal[i], i, dt)
full_change = total_action(proposal, dt) - total_action(example_path, dt)
print(f"Local change: {local_change:.8f}")
print(f"Full change:  {full_change:.8f}")
np.testing.assert_allclose(local_change, full_change, atol=1e-12)

Agreement here checks the action calculation. It does not yet show that a Markov chain has equilibrated.

## Accept or reject

The proposal is symmetric, so the Metropolis acceptance probability is

$$A=\min\left(1,e^{-\Delta S_E/\hbar}\right).$$

Accept every move that lowers the action. Sometimes accept a move that raises it. A rejected move leaves the path unchanged and still counts as an attempt.

## One Metropolis update

The oscillator has no hard position walls. We let the path explore the real line.

In [ ]:
@njit
def update_point(path, i, dt, hitsize, rng):
    x_prime = path[i] + rng.uniform(-hitsize, hitsize)
    change = delta_action(path, x_prime, i, dt)
    if change <= 0 or rng.random() < np.exp(-change / HBAR):
        path[i] = x_prime
        return 1
    return 0

Plot limits only control the picture. Redrawing a proposal until it lies inside the plotting window would change its probability near the edges.

## One sweep

Visit every stored point once, in order. Each accepted change is immediately used by the following update.

In [ ]:
@njit
def sweep(path, dt, hitsize, rng):
    accepted = 0
    for i in range(path.size):
        accepted += update_point(path, i, dt, hitsize, rng)
    return accepted / path.size

A fixed sequential sweep is valid here. One sweep produces the next whole-path configuration; successive paths are correlated.

## Warm up, then collect measurements

Start from zeros. Keep a trace of the warm-up, but exclude it from the histogram and reported average.

In [ ]:
@njit
def sample_paths(n_tau, dt, hitsize, burn_in, sweeps, rng):
    path = np.zeros(n_tau)
    warmup = np.empty(burn_in)
    samples = np.empty((sweeps, n_tau))
    acceptance = np.empty(sweeps)
    for k in range(burn_in):
        sweep(path, dt, hitsize, rng)
        warmup[k] = np.mean(path**2)
    for k in range(sweeps):
        acceptance[k] = sweep(path, dt, hitsize, rng)
        samples[k] = path
    return warmup, samples, acceptance

Each row of `samples` is one retained path. Copying it into a row preserves that configuration before the next sweep changes `path`.

## Run the coarse time grid

Begin with the original lab's $\Delta\tau=1$ and total imaginary time 30. A wider proposal helps the path explore faster than the original $h=0.1$.

In [ ]:
DT = 1.0
NTAU = int(round(TAU / DT))
HITSIZE = 1.0
BURN_IN = 10_000
SWEEPS = 40_000
warmup, samples, acceptance = sample_paths(
    NTAU, DT, HITSIZE, BURN_IN, SWEEPS, np.random.default_rng(142))
x2_trace = np.mean(samples**2, axis=1)
print(f"{samples.shape[0]:,} paths; {samples.shape[1]} points per path")
print(f"Accepted fraction: {acceptance.mean():.3f}")

The first call includes compilation. Rerunning this cell restarts the chain from the same seed; it does not accumulate old measurements.

## Watch the chain

For each path, measure $\overline{x^2}=N_\tau^{-1}\sum_i x_i^2$. This is one measurement per sweep.

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(BURN_IN), warmup, lw=0.5, color="#a54632", label="Discarded")
ax.plot(BURN_IN + np.arange(SWEEPS), x2_trace,
        lw=0.5, color="#007c91", label="Retained")
ax.axvline(BURN_IN, color="black", ls="--", lw=1)
ax.set(xlabel="Sweep", ylabel=r"Path mean of $x^2$")
ax.legend()
plt.show()

Look for drift and long correlated stretches. A trace is a useful diagnostic, not proof of equilibrium; production work should compare starts and run lengths.

## Several sampled paths

The chain samples whole paths. The position histogram combines the coordinates from many retained paths.

In [ ]:
fig, ax = plt.subplots()
for k in [0, SWEEPS // 2, SWEEPS - 1]:
    closed = np.r_[samples[k], samples[k, 0]]
    ax.plot(np.arange(NTAU + 1) * DT, closed, lw=1, label=f"Path {k}")
ax.set(xlabel=r"Imaginary time $\tau$", ylabel="Position x")
ax.legend()
plt.show()

Coordinates within one path are correlated too. Counting all coordinates as independent samples would underestimate the uncertainty.

## The finite grid has its own answer

For a long harmonic-oscillator path with this left-endpoint action,

$$\langle x^2\rangle_{\Delta\tau}\simeq
\frac{\hbar}{m\omega\sqrt{4+(\omega\Delta\tau)^2}}.$$

At $\Delta\tau=1$, this is **0.4472**, not 0.5. The exact finite-path result can also be found by summing its normal modes.

In [ ]:
def lattice_variance(n_tau, dt):
    modes = np.arange(n_tau)
    eigenvalues = dt * OMEGA**2 + 4 / dt * np.sin(np.pi * modes / n_tau)**2
    return HBAR / M * np.mean(1 / eigenvalues)

variance_dt = lattice_variance(NTAU, DT)
print(f"Finite-grid prediction: {variance_dt:.6f}")

This is a reference calculation, not part of the sampler. Keeping the left-endpoint potential makes the time-step error visible.

## Compare probability densities

First compare with the distribution the finite grid actually samples. Then compare with the continuum ground state.

In [ ]:
rho_lattice = np.exp(-x_grid**2 / (2 * variance_dt)) / np.sqrt(
    2 * np.pi * variance_dt)
fig, ax = plt.subplots()
ax.hist(samples.ravel(), bins=100, density=True, color="#9fc9d3", label="MCMC")
ax.plot(x_grid, rho_lattice, color="#172b4d", lw=2, label="Finite grid")
ax.plot(x_grid, rho_ground, "--", color="#a54632", lw=2, label="Continuum")
ax.set(xlim=(-3, 3), xlabel="Position x", ylabel="Probability density")
ax.legend()
plt.show()

A narrower histogram at this time step is expected. More sweeps reduce sampling noise; they do not remove this discretization error.

## Put an uncertainty on the width

Use the per-sweep measurements and the blocking method from Lab 12. Neighboring blocks must be long enough to be approximately independent.

In [ ]:
def blocked_mean_error(values, block_size):
    n_blocks = values.size // block_size
    blocks = values[:n_blocks * block_size].reshape(n_blocks, block_size)
    means = blocks.mean(axis=1)
    return values.mean(), means.std(ddof=1) / np.sqrt(n_blocks)

for block_size in (250, 500, 1000, 2000):
    mean, error = blocked_mean_error(x2_trace, block_size)
    print(f"Block {block_size:4d}: <x²> = {mean:.4f} ± {error:.4f}")

Look for an error estimate that settles as the block size grows, while retaining enough blocks. If it keeps growing, collect a longer chain.

## Reduce the time step

Keep the total imaginary time fixed. Halving the step doubles the number of coordinates in each path.

In [ ]:
time_steps = np.array([1.0, 0.5, 0.25])
means, errors = [blocked_mean_error(x2_trace, 1000)[0]], [
    blocked_mean_error(x2_trace, 1000)[1]]
for run, smaller_dt in enumerate(time_steps[1:], start=1):
    smaller_n = int(round(TAU / smaller_dt))
    _, fine_samples, _ = sample_paths(
        smaller_n, smaller_dt, HITSIZE * np.sqrt(smaller_dt),
        BURN_IN, SWEEPS, np.random.default_rng(142 + run))
    mean, error = blocked_mean_error(np.mean(fine_samples**2, axis=1), 1000)
    means.append(mean)
    errors.append(error)

Scaling the proposal with $\sqrt{\Delta\tau}$ keeps the kinetic-action change comparable. We still need to check mixing as the time grid gets finer.

## Separate the two errors

The error bars estimate sampling uncertainty. The curve shows the bias from the finite time step.

In [ ]:
dt_curve = np.linspace(0, 1.05, 200)
variance_curve = HBAR / (M * OMEGA * np.sqrt(4 + (OMEGA * dt_curve)**2))
fig, ax = plt.subplots()
ax.plot(dt_curve, variance_curve, color="#172b4d", label="Long-path grid prediction")
ax.axhline(0.5, color="#a54632", ls="--", label="Continuum ground state")
ax.errorbar(time_steps, means, yerr=errors, fmt="o", capsize=4,
            color="#007c91", label="MCMC ± blocked error")
ax.set(xlabel=r"Time step $\Delta\tau$", ylabel=r"$\langle x^2\rangle$")
ax.legend(fontsize=10)
plt.show()

Agreement with the grid prediction checks the sampler. Convergence toward 0.5 checks the time discretization. The finer chain is more correlated, so its blocked error estimate is noisier.

## What the run controls do

| Change | What it addresses |
|---|---|
| More retained sweeps | Sampling uncertainty, once equilibrated |
| Longer discarded warm-up | Sensitivity to the starting path |
| Different proposal size | Acceptance and exploration speed |
| Smaller $\Delta\tau$ at fixed total time | Time-discretization error |
| Longer total imaginary time | Excited-state contributions |

An acceptance fraction alone does not tell us whether the chain explores well.

## Carry this into Assignment 3

Follow the assignment's specified total imaginary time and action. At finite total time, compare with the thermal result, not automatically with the ground state:

$$\langle x^2\rangle=\frac{\hbar}{2m\omega}\coth\!\left(\frac{\omega\tau_{\rm total}}2\right).$$

For each result, state the time step, run length, discarded warm-up, and uncertainty method. Check the sampler against the finite grid before interpreting a continuum discrepancy.

## The workflow we used

1. Define the path and its action.
2. Check a local action change against a full recomputation.
3. Sample with symmetric proposals and a stated warm-up.
4. Measure the density and $\langle x^2\rangle$ with correlated errors.
5. Reduce the time step and compare with the known answer.

Adapted from Javier Duarte's Lab 14 and the Mittal et al. example discussed in lecture. The left-endpoint potential follows the corrected course notebook.